In [1]:
#Code 5 (reads in master_df_with_features_clusters and appends SVI data)
#df is the dataframe where output is ultimately stored and saved as master_df_with_features_clusters_SVI.xlsx

%matplotlib inline
import pandas as pd
import numpy as np


In [3]:
df = pd.read_excel('output/master_df_with_features_clusters.xlsx')
#df = pd.read_excel('output/master_df_with_features_clusters_v2_(exclude cluster3).xlsx')
SVI = pd.read_excel('data/SVI_2022_US_county.xlsx')

df=df.drop(columns=['count_age_55_64','count_age_65_plus','count_gt_400_fpl','benchmark_60_401_diff','count_55_plus','factor_age_55_plus','raw_risk_cliff'])

In [4]:
svi_features_keep = [
    'FIPS',        # Join Key
    # EP = est. percentage
    # Tier 1: Barriers (Physical/Cultural)
    'EP_NOVEH',    # No Vehicle (Access barrier)
    'EP_LIMENG',   # Limited English (Navigational barrier)
    'EP_DISABL',   # Disability (High resilience risk)
    'EP_SNGPNT',   # Single Parent (Bandwidth/Time tax)
    # Tier 2: Financial Stressors
    'EP_HBURD',    # Housing Burden (>30% income on housing)
    'EP_POV150',   # 150% Poverty (Broader deprivation measure)
    'EP_UNEMP'     # Unemployment
]

# Check which columns are actually in the loaded SVI file
available_svi_cols = [c for c in svi_features_keep if c in SVI.columns]
missing_svi_cols = [c for c in svi_features_keep if c not in SVI.columns]

if missing_svi_cols:
    print(f"⚠️ Warning: The following desired SVI columns were missing: {missing_svi_cols}")

# B. Create the Clean SVI DataFrame
SVI_clean = SVI[available_svi_cols].copy()

# C. Standardize FIPS for Join
# SVI 'FIPS' is often an Integer (e.g., 1001). We need String "01001".
if 'FIPS' in SVI_clean.columns:
    SVI_clean['FIPS_Join'] = SVI_clean['FIPS'].astype(str).str.zfill(5)
    # Drop original raw FIPS to avoid confusion
    SVI_clean = SVI_clean.drop(columns=['FIPS'])
else:
    raise KeyError("CRITICAL: 'FIPS' column not found in SVI dataframe.")

print(f"✅ SVI Cleaned. Retained {len(SVI_clean.columns)-1} features.")

✅ SVI Cleaned. Retained 7 features.


In [5]:
# ---------------------------------------------------------
# 3. Merge Logic
# ---------------------------------------------------------
print(f"\n--- 🔗 Merging SVI into Master DataFrame ---")

# Identify the FIPS column in 'df' (Handling variations)
possible_fips_cols = ['county_fips', 'County FIPS Code', 'fips', 'FIPS']
left_key = None

for col in possible_fips_cols:
    if col in df.columns:
        left_key = col
        break

if left_key:
    # Ensure left key is standardized (String, 5 chars)
    df[left_key] = df[left_key].astype(str).str.zfill(5)
    
    # Perform Merge
    # how='left' -> Keep all counties in your master file, even if SVI is missing
    df_merged = df.merge(SVI_clean, left_on=left_key, right_on='FIPS_Join', how='left')
    
    # Clean up artifacts
    df_merged = df_merged.drop(columns=['FIPS_Join'])
    
    # Update the main variable
    df = df_merged
    
    print(f"✅ Merge Complete.")
    print(f"   Original Rows: {len(df)}")
    print(f"   Merged Rows:   {len(df_merged)} (Should match)")
    print(f"   New Columns Added: {[c for c in available_svi_cols if c != 'FIPS']}")
else:
    print("❌ CRITICAL ERROR: Could not find a FIPS column in 'df'. Merge failed.")
    print(f"   Available columns: {df.columns.tolist()}")

# ---------------------------------------------------------
# 4. Verification & Output
# ---------------------------------------------------------
print("\n--- Final Data Structure ---")
print(df.info())

df.to_excel('output/master_df_with_features_clusters_SVI.xlsx', index=False)
#df.to_excel('output/master_df_with_features_clusters_SVI_v2_(exclude cluster3).xlsx', index=False)





--- 🔗 Merging SVI into Master DataFrame ---
✅ Merge Complete.
   Original Rows: 2156
   Merged Rows:   2156 (Should match)
   New Columns Added: ['EP_NOVEH', 'EP_LIMENG', 'EP_DISABL', 'EP_SNGPNT', 'EP_HBURD', 'EP_POV150', 'EP_UNEMP']

--- Final Data Structure ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2156 entries, 0 to 2155
Data columns (total 53 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   County FIPS Code                  2156 non-null   object 
 1   State                             2156 non-null   object 
 2   total_enrollees_master            2156 non-null   int64  
 3   count_gt_100_150_fpl              2156 non-null   int64  
 4   count_gt_150_200_fpl              2156 non-null   int64  
 5   V1_lowincome                      2156 non-null   float64
 6   V2_ooprisk                        2156 non-null   float64
 7   V3_Subsidycliff                   2156 non-null   float6